# Part 5 - Learning from Failure: in-loop reflection and Reflexion

*Agents from First Principles, Part 5.*

Part 4 taught the agent to revise a plan when the world disagrees mid-run. But run the same agent twice on the same task and it makes the same mistake the same way both times. Replanning recovers WITHIN a run; nothing carries WHAT IT LEARNED from one run to the next. The agent is an amnesiac that re-earns every lesson.

This part gives it two ways to learn from a failure, one within a single attempt and one across attempts.

1. **IN-LOOP SELF-CRITIQUE.** Before it finishes, the agent re-reads its own draft against a checklist and fixes what it can SEE: a missing amount, an uncited policy. Cheap, and it catches presentation mistakes in the same trial. But it has a ceiling: the agent cannot self-critique a WRONG NUMBER it has no ground truth for. It will confidently polish a draft that is quietly incorrect.
2. **REFLEXION** (Shinn et al., 2023). The mistake that self-critique cannot see is caught by an external **CHECKER**, a reward signal from the environment. Reflexion turns that verdict into a VERBAL post-mortem and writes it to an **EPISODIC BUFFER**, and the next attempt READS the buffer before it acts. The loop is `actor -> checker (reward) -> self-reflection (verbal) -> buffer -> actor ...`. This is **verbal reinforcement**: the policy is not retrained, the lesson is just text the next trial reads.

**THE HONEST PART** (this is the whole point, do not fake it). The deterministic reflection here is a REAL rule whose output the actor MECHANICALLY reads on the next trial. Convergence is genuinely CAUSED by the buffer, not by a script that flips trial 2 to "correct." We prove it by running the same loop with the buffer TURNED OFF: with no memory, the agent repeats the identical mistake every trial and never converges. Same actor, same task; the only difference is whether the lesson persists.

A sober note carried throughout: reflection is not monotonic. A wrong reflection can mislead the next trial and make it WORSE. Verbal reinforcement is a heuristic, not a guarantee.

**THE TASK.** A return for order `ORD-3300`, a `$200` order, comes in AFTER the 30-day window. Policy: a return after the window still gets a refund, but a 10% restocking fee applies, so the correct refund is `$180.00`, and the decision must state the amount and the policy basis. The naive actor forgets the fee and quotes `$200.00`.

**CONNECTION**: the CHECKER is RAG Part 11's LLM-as-judge repurposed as an offline reward signal (does the output satisfy the policy?), not answer-quality scoring. The episodic buffer here is the SEED of the typed episodic memory store Part 6 formalizes.

> **Runs with no network, no API key, and no third-party dependency.** Set `OPENAI_API_KEY` to see the real-actor banner; the actor, checker, and reflector always fall through to the deterministic rules so output stays reproducible. Same refund world as Parts 1-4.

Companion script: `reflexion.py`

## Setup

One standard-library import does the work: `os` lets us check for an API key without ever requiring one. Nothing in the default path is imported from a third-party package, so every cell runs fully offline, exactly as in Parts 1-4.

In [ ]:
import os

print("ready -- no network, no API key, no third-party dependency required")

## Step 0 - The task and its policy-correct answer

The restocking fee is the trap: a return after the 30-day window is still refundable, but at 90% of the order total. We pin the four facts the rest of the notebook keys on: the order id, its `$200.00` total, the 10% fee, and the `$180.00` the policy actually allows. `CORRECT_REFUND` is what the external checker knows and the naive actor does not.

In [ ]:
ORDER_ID = "ORD-3300"
ORDER_TOTAL = 200.0
RESTOCKING_FEE = 0.10                     # 10% for returns after the 30-day window
CORRECT_REFUND = round(ORDER_TOTAL * (1 - RESTOCKING_FEE), 2)   # 180.00

print(f"order {ORDER_ID}: total ${ORDER_TOTAL:.2f}, policy-correct refund ${CORRECT_REFUND:.2f}")

## Step 1 - The actor, whose computation READS the reflection buffer

This is the mechanical link that makes the whole part honest. `actor_compute` decides the refund by looking at the reflections it was handed: if any reflection mentions the restocking fee, it applies the fee and quotes `$180.00`; otherwise it quotes the full `$200.00`. The actor behaves differently ONLY because of what it read. There is no branch on the trial number, no script that flips trial 2 to correct. Convergence, when it happens, is caused by the contents of the buffer.

`actor_draft` is the naive first draft: a bare approval, missing the amount and the basis. That is the presentation gap in-loop self-critique will fix in the next step.

In [ ]:
def actor_compute(order_total, reflections):
    learned_fee = any("restocking fee" in r.lower() for r in reflections)
    if learned_fee:
        refund = round(order_total * (1 - RESTOCKING_FEE), 2)
        note = "applied the 10% restocking fee (learned from a reflection)"
    else:
        refund = order_total
        note = "full refund (no restocking fee considered)"
    return refund, note


def actor_draft():
    """The naive first draft: a bare approval, missing the amount and the basis."""
    return "Your return is approved."


print("actor_compute and actor_draft ready.")

## Step 2 - In-loop self-critique: fix the draft you can SEE, within one trial

Within a single trial, re-read the draft against a checklist and fix what is visible: state the refund amount, cite the policy basis. This is cheap and it catches real presentation mistakes in the same attempt, no extra trial needed.

Note what it CANNOT do, because that ceiling is the reason Reflexion exists. Self-critique has no ground truth for the number; it only checks that the draft *mentions* an amount, not that the amount is *right*. It will happily polish a draft that quotes `$200.00` into a complete, well-cited sentence that is still wrong.

In [ ]:
def self_critique(draft, refund):
    issues = []
    if f"${refund:.2f}" not in draft:
        issues.append("state the refund amount")
    if "policy" not in draft.lower():
        issues.append("cite the policy basis")
    if not issues:
        return draft, issues
    revised = (f"Refund decision for {ORDER_ID}: ${refund:.2f} approved, per the "
               "returns policy (item returned after the 30-day window).")
    return revised, issues


print("self_critique ready.")

## Step 3 - The external CHECKER: the reward signal

The checker knows the policy-correct refund (the environment does), so it can catch the wrong number self-critique cannot. It returns a pass/fail verdict plus a reason, and on failure the reason names the actual cause: the 10% restocking fee was not applied. That reason is what the reflector will turn into a lesson.

This is RAG Part 11's LLM-as-judge **repurposed**: there it scored answer quality, here it is a pass/fail oracle against a policy. It is the reward signal of the Reflexion loop, not a quality grade.

In [ ]:
def checker(refund):
    if abs(refund - CORRECT_REFUND) < 0.01:
        return True, f"refund ${refund:.2f} matches the policy-correct ${CORRECT_REFUND:.2f}"
    return (False, f"refund ${refund:.2f} != policy-correct ${CORRECT_REFUND:.2f}: the 10% "
            "restocking fee for returns after the window was not applied")


print("checker ready.")

## Step 4 - The reflector: turn a verdict into an actionable lesson

The reflector takes the checker's failure reason and writes a concrete, actionable verbal lesson the actor can read next time. The offline version is a real rule (it keys on the failure reason); a real `generate()` would write the same note in prose. The lesson MUST be specific enough to change behavior: "multiply the order total by 0.9" is something `actor_compute` can act on, where a vague "be more careful" is not.

The fallback branch is the sober note made concrete: if the reflector cannot pin the cause it emits a generic "try a different approach," which may or may not help the next trial. A reflection is a heuristic, not a guarantee.

In [ ]:
def reflect(checker_reason):
    if "restocking fee" in checker_reason:
        return ("Returns after the 30-day window incur a 10% restocking fee; multiply "
                "the order total by 0.9 before quoting the refund.")
    return f"Trial failed: {checker_reason}. Try a different approach."


print("reflect ready.")

## Step 5 - `generate()`: the real LLM path (reference shape only)

Same device as Parts 1-4. In production the actor, checker, and reflector are all an LLM: you hand it the task, the draft, or the failure, and it acts, grades, or writes the post-mortem. `generate()` is the single swappable call that lights up the real path; the offline demo never calls it. The deterministic rule actor, checker, and reflector are the source of truth for everything you see in the output. SDK names and model IDs move fast, so only `generate()` would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: ask a hosted LLM to act, check, or reflect. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[actor] OPENAI_API_KEY set; the real LLM would act/check/reflect via "
          "generate(). Falling through to the deterministic rules so output is reproducible.")
else:
    print("[actor] no OPENAI_API_KEY; using deterministic rule actor/checker/reflector "
          "(offline default)")

## Step 6 - The Reflexion loop, with the buffer as the one switch

Each trial runs the full cycle: the actor (reading the buffer) computes and drafts, self-critique fixes the draft, the checker grades. On a PASS the loop converges and returns. On a FAIL the reflector writes a lesson, and here is the only difference between the two runs that follow:

- with the buffer **ON** (`use_buffer=True`), the lesson is appended to `reflections`, and the next trial's `actor_compute` reads it;
- with the buffer **OFF**, the lesson is discarded, and the next trial starts amnesiac with an empty buffer.

Same actor, same checker, same reflector, same task. The buffer is the single switch, which is exactly what lets the control in the demo isolate it as the cause of convergence.

In [ ]:
def reflexion_run(max_trials=3, use_buffer=True):
    reflections = []
    final = None
    for trial in range(1, max_trials + 1):
        print(f"  Trial {trial} (reflections in buffer: {len(reflections)})")
        refund, compute_note = actor_compute(ORDER_TOTAL, reflections)
        print(f"    actor: {compute_note} -> ${refund:.2f}")
        draft = actor_draft()
        revised, issues = self_critique(draft, refund)
        if issues:
            print(f"    self-critique: revise ({'; '.join(issues)})")
            print(f"      -> {revised!r}")
        else:
            print(f"    self-critique: no issues")
        final = revised
        ok, reason = checker(refund)
        if ok:
            print(f"    checker: PASS -- {reason}")
            print(f"  -> converged on trial {trial}"
                  + (" (the buffer carried the lesson forward)" if use_buffer and trial > 1 else ""))
            return trial, final
        print(f"    checker: FAIL -- {reason}")
        note = reflect(reason)
        if use_buffer:
            reflections.append(note)
            print(f"    reflection appended: {note!r}")
        else:
            print(f"    buffer OFF: reflection discarded; next trial starts amnesiac")
    print(f"  -> did NOT converge in {max_trials} trials")
    return None, final


print("reflexion_run ready.")

## Demo - the self-critique ceiling, then buffer ON, then the buffer-OFF control

Everything below runs fully offline. The demo has three sections. First, **in-loop self-critique** on the naive draft: it makes the text complete but the external checker still fails it, because self-critique cannot catch a wrong number. Second, **Reflexion with the buffer ON**: the checker's verdict becomes a lesson trial 2 reads, and the agent converges. Third, **the control with the buffer OFF**: the same actor on the same task, but the lesson is discarded every trial, and it never converges.

We begin by printing the task, then run self-critique with an EMPTY buffer (no reflections yet), so it quotes the naive `$200.00`. Self-critique fills in the missing amount and the policy basis, producing a complete, well-formed decision. But when the external checker grades the underlying number, it still FAILS: a polished `$200.00` is just as wrong as a bare one. Self-critique made the text complete; it could not catch a wrong number it has no ground truth for. That gap is precisely what Reflexion fills.

In [ ]:
bar = "=" * 72
print(bar)
print("LEARNING FROM FAILURE  -  in-loop self-critique and cross-trial Reflexion")
print(bar)
if os.environ.get("OPENAI_API_KEY"):
    print("[actor] OPENAI_API_KEY set; the real LLM would act/check/reflect via "
          "generate(). Falling through to the deterministic rules so output is reproducible.")
else:
    print("[actor] no OPENAI_API_KEY; using deterministic rule actor/checker/reflector "
          "(offline default)")
print(f"\nTASK: refund decision for {ORDER_ID}, a ${ORDER_TOTAL:.2f} order returned AFTER")
print(f"the 30-day window. Policy-correct refund is ${CORRECT_REFUND:.2f} (a 10% restocking fee).")

print("\n" + "-" * 72)
print("IN-LOOP SELF-CRITIQUE: fix the draft you can see (within one trial).")
print("-" * 72)
refund0, note0 = actor_compute(ORDER_TOTAL, [])          # no reflections yet
draft0 = actor_draft()
print(f"  draft:  {draft0!r}")
revised0, issues0 = self_critique(draft0, refund0)
print(f"  critique: {'; '.join(issues0)}")
print(f"  revised: {revised0!r}")
ok0, reason0 = checker(refund0)
print(f"  ...but the external checker still says: {'PASS' if ok0 else 'FAIL'} -- {reason0}")
print("  -> self-critique made the text complete; it could NOT catch a wrong number")
print("     it has no ground truth for. That gap is what Reflexion fills.")

### Reflexion with the buffer ON: the lesson persists across trials

Now run the full loop with the buffer ON. Trial 1 has an empty buffer, so the actor quotes `$200.00`, the checker fails it, and the reflector appends the restocking-fee lesson. Trial 2's `actor_compute` reads that lesson, applies the fee, quotes `$180.00`, and the checker passes. It converges on trial 2, and it converges because the buffer carried the lesson forward, not because anything in the code knew it was trial 2.

In [ ]:
print("\n" + bar)
print("REFLEXION (buffer ON): the checker's verdict becomes a lesson the next trial reads.")
print(bar)
trial_on, ans_on = reflexion_run(max_trials=3, use_buffer=True)

### The control with the buffer OFF: this is what proves causation

This cell is the load-bearing one. We run the SAME actor on the SAME task with the SAME reflector, changing exactly one thing: the buffer is OFF, so each trial's lesson is discarded and the next trial starts amnesiac. The result is that the identical `$200.00` mistake repeats on trials 1, 2, and 3, and the agent never converges.

Because the only difference between this run and the buffer-ON run is whether the lesson persists, the buffer is what CAUSED convergence above, not a script that quietly flipped trial 2 to correct. If convergence had been scripted, it would happen here too. It does not. This control is the proof that the appended reflection mechanically changed the actor's computation.

In [ ]:
print("\n" + bar)
print("THE CONTROL (buffer OFF): same actor, same task, but the lesson is discarded.")
print(bar)
trial_off, ans_off = reflexion_run(max_trials=3, use_buffer=False)

### What just happened, and a sober note

Side by side: buffer ON converged on trial 2, the appended reflection changed the actor's computation from `$200.00` to `$180.00`; buffer OFF never converged, the identical mistake repeated every trial. The only difference was whether the lesson persisted, so the BUFFER, not a script, is what caused convergence. That episodic buffer is the seed of Part 6's typed episodic memory.

And the sober note, because it matters: reflection is not monotonic. A wrong lesson can mislead the next trial and make it WORSE. Verbal reinforcement is a heuristic, not a guarantee. The fact that it converged here is a property of a good reflection, not a property you get for free.

In [ ]:
print("\n" + bar)
print("WHAT JUST HAPPENED")
print(bar)
print(f"  buffer ON : converged on trial {trial_on} -- the appended reflection changed the")
print(f"              actor's computation from ${ORDER_TOTAL:.2f} to ${CORRECT_REFUND:.2f}.")
print(f"  buffer OFF: never converged -- the identical mistake repeated every trial.")
print("  The only difference was whether the lesson persisted, so the BUFFER (not a")
print("  script) is what caused convergence. This is the seed of Part 6's episodic memory.")
print("  Sober note: reflection is not monotonic. A wrong lesson can mislead the next")
print("  trial and make it worse; verbal reinforcement is a heuristic, not a guarantee.")
print(bar)

## Wrap-up

Part 4 recovered within a run, but ran twice it repeated the same mistake, because nothing persisted what it learned. This part added two ways to learn from a failure:

- **in-loop self-critique**, which re-reads the draft against a checklist and fixes what it can see (a missing amount, an uncited policy), cheaply and within one trial, but cannot catch a wrong number it has no ground truth for;
- **Reflexion**, where an external CHECKER (the reward signal, RAG Part 11's judge repurposed) catches that wrong number, the reflector turns the verdict into a VERBAL lesson written to an EPISODIC BUFFER, and the next trial reads the buffer before acting. No retraining: the lesson is just text.

The buffer-OFF control was the whole point: same actor, same task, lesson discarded, and it never converged. That isolates the buffer as the genuine cause of convergence, not a script. And the sober note stands: a wrong reflection can make the next trial worse, so verbal reinforcement is a heuristic, not a guarantee.

**Part 6 - Memory** is next. The reflection buffer here was a flat list of strings that lives only for the length of one `reflexion_run`. Part 6 formalizes it into a typed, persistent **episodic memory** store the agent writes to and reads from across runs, alongside the other kinds of memory an agent needs.